In [0]:
%sql
CREATE VOLUME IF NOT EXISTS rapidroute.silver.checkpoints;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS rapidroute.silver.shipment_current_state (
    shipment_id             STRING NOT NULL,
    carrier_id              STRING,
    status                  STRING,
    service_level           STRING,
    origin_city             STRING,
    destination_city        STRING,
    event_timestamp_parsed  TIMESTAMP,
    estimated_delivery_ts_parsed TIMESTAMP,
    weight_kg               FLOAT,
    customer_id             STRING,
    _last_updated           TIMESTAMP
)
USING DELTA;

In [0]:
# stream_clean_events verification

from pyspark.sql.functions import col
silver_clean = spark.read.table("rapidroute.silver.shipment_events_clean")

# Check for any UNKNOWN statuses
silver_clean.filter(col("status") == "UNKNOWN").count()

# Check timestamp parsing succeeded
silver_clean.select(
    "shipment_id", "status",
    "event_timestamp_parsed", "estimated_delivery_ts_parsed"
).show(5, truncate=False)

# Confirm no nulls on critical fields
from pyspark.sql.functions import count, when, isnan
silver_clean.select(
    count(when(col("shipment_id").isNull(), 1)).alias("null_shipment_ids"),
    count(when(col("event_timestamp_parsed").isNull(), 1)).alias("null_timestamps")
).show()

In [0]:
# stream_current_state verification

from pyspark.sql.functions import col

current_state = spark.read.table("rapidroute.silver.shipment_current_state")

print(f"Distinct shipments: {current_state.count()}")

# Confirm no duplicates — should be exactly one row per shipment_id
from pyspark.sql.functions import count
current_state.groupBy("shipment_id").count().filter(col("count") > 1).show()

# Status distribution
current_state.groupBy("status").count().orderBy("count", ascending=False).show()

# Verifying flow with new batch of data
- Manually craft an event with the same shipment ID but a newer timestamp and status and inject it directly into the pipeline 
- If the MERGE is working correctly you'll see exactly one row with status = delivered

In [0]:
import json
from datetime import datetime, timedelta
from pyspark.sql.functions import col, rand

# Grab the shipment you want to test
shipment = (spark.read.table("rapidroute.silver.shipment_current_state")
            .filter(col("status") == "CREATED")
            .orderBy(rand())
            .first())

spark.read.table("rapidroute.silver.shipment_current_state") \
    .filter(col("shipment_id") == shipment["shipment_id"])

# Craft a new event for the same shipment with a newer timestamp and a forced status
new_event = {
    "event_id":             "test-event-001",
    "shipment_id":          shipment["shipment_id"],
    "carrier_id":           shipment["carrier_id"],
    "status":               "delivered",        # force a specific status change
    "event_timestamp":      (datetime.utcnow() + timedelta(hours=1)).isoformat(),  # newer than existing
    "origin_city":          shipment["origin_city"],
    "destination_city":     shipment["destination_city"],
    "service_level":        shipment["service_level"],
    "estimated_delivery_ts": datetime.utcnow().isoformat(),
    "weight_kg":            shipment["weight_kg"],
    "customer_id":          shipment["customer_id"]
}

# Write it as a new JSON file directly to the Volume
output_path = "/Volumes/rapidroute/bronze/raw_events/test_event_001.json"
with open(output_path, "w") as f:
    f.write(json.dumps(new_event))

print(f"Injected test event for {new_event['shipment_id']} with status: {new_event['status']}")

# Run the pipeline
exec(open("../bronze/ingest_events.py").read())
exec(open("./stream_clean_events.py").read())
exec(open("./stream_current_state.py").read())

# Check result — should show 'delivered', still only one row
display(spark.read.table("rapidroute.silver.shipment_current_state") \
    .filter(col("shipment_id") == new_event["shipment_id"]))

# Reverse Flow
- Inject an event with an older timestamp and confirm the status does not change.
- If the second test still shows delivered, your MERGE condition is correct

In [0]:
# Now inject an older event — status should NOT update
old_event = {**new_event,
    "event_id":        "test-event-002",
    "status":          "in_transit",   # an earlier status
    "event_timestamp": (datetime.utcnow() - timedelta(hours=5)).isoformat()  # older timestamp
}

with open("/Volumes/rapidroute/bronze/raw_events/test_event_002.json", "w") as f:
    f.write(json.dumps(old_event))

exec(open("../bronze/ingest_events.py").read())
exec(open("./stream_clean_events.py").read())
exec(open("./stream_current_state.py").read())

# Should still show 'delivered' — older event must not overwrite newer state
display(spark.read.table("rapidroute.silver.shipment_current_state") \
    .filter(col("shipment_id") == new_event["shipment_id"]))